# Recertification KYC — Traitement d'un ZIP multi-clients (extraction sans schéma fixe)

## Nouveau contexte
- Une **archive ZIP** contient un dossier par client, **nommé par son ID**
- Chaque dossier client contient **plusieurs PDF**, dont on ne traite que deux :
  - `carte identite.pdf`
  - `residence.pdf`
- Documents multi-pays, multi-formats, **multi-langues**
- **Contrainte forte** : pas de schéma JSON figé par type de document (le pipeline précédent définissait un schéma `id_card` et un schéma `residence_permit` — ce n'est plus utilisé ici)

## Changements clés vs. la version précédente

| Avant | Maintenant |
|---|---|
| Images uniques en entrée | **ZIP → dossiers clients → PDF** (conversion PDF→images incluse, multi-pages) |
| Schéma JSON fixe par type de document | **Extraction libre "clé → valeur"**, les champs sont ceux réellement présents sur le document, sans liste prédéfinie |
| Un seul document traité | **Deux documents par client**, avec **cross-vérification d'identité** entre les deux (le nom sur la carte d'identité doit correspondre à celui sur le justificatif de résidence) |
| OCR mono-langue supposé | **Sélection automatique du pack linguistique PaddleOCR** (latin / arabe / cyrillique / CJK) par essai-comparaison de confiance |

## Architecture du pipeline

1. **Décompression** du ZIP + parcours des dossiers clients (nom de dossier = ID client)
2. **Repérage flexible** des deux fichiers cibles dans chaque dossier (tolérant à la casse, accents, espaces/underscores)
3. **Conversion PDF → images** (PyMuPDF), page par page (recto/verso, documents multi-pages)
4. **Prétraitement image** : rotation grossière + skew fin + débruitage/contraste (repris du notebook précédent)
5. **OCR multi-langue adaptatif** (PaddleOCR) : essai de plusieurs packs linguistiques, on garde celui de plus haute confiance — sert de score de qualité, pas d'extraction de champs
6. **Extraction schema-free** par Qwen2.5-VL et InternVL : sortie JSON à structure libre `{"champs": [{"libelle": ..., "valeur": ...}, ...]}`, sans schéma imposé
7. **Réconciliation floue** (fuzzy matching) entre les deux VLM sur les libellés de champs
8. **Cross-vérification d'identité** entre `carte identite.pdf` et `residence.pdf` du même client
9. **Traitement par lot** sur tous les clients du ZIP → export JSON/CSV


In [ ]:
import os
import re
import io
import json
import zipfile
import shutil
import unicodedata
import difflib
from pathlib import Path
from collections import defaultdict
from typing import Optional

import cv2
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
from PIL import Image
import torch


## 1. Configuration : chemins ModelHub et archive ZIP à traiter

In [ ]:
MODEL_PATHS = {
    "qwen_vlm": "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main",
    "internvl": "/domino/edv/modelhub/ModelHub-model-huggingface-OpenGVLab/InternVL2_5-8B/main",
}

ZIP_PATH = "/mnt/user-data/uploads/dossiers_clients.zip"   # à adapter
EXTRACT_DIR = "/home/work/kyc_extraction"                  # dossier de travail (scratch)
OUTPUT_DIR = "/mnt/user-data/outputs"

# Noms de fichiers cibles (comparaison normalisée : minuscule, sans accents, espaces/underscores ignorés)
TARGET_FILES = {
    "id_card": ["carteidentite", "carte identite", "cni", "idcard"],
    "residence": ["residence", "justificatifdomicile", "residencepermit"],
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {DEVICE}")
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Décompression du ZIP et découverte des dossiers clients

Chaque sous-dossier de premier niveau de l'archive correspond à un client, nommé par son ID.


In [ ]:
def extract_zip(zip_path: str, dest_dir: str) -> str:
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    return dest_dir


def list_client_folders(root_dir: str) -> dict:
    """Retourne {client_id: chemin_dossier}. Gère le cas où le zip contient
    un dossier racine unique englobant tous les dossiers clients."""
    entries = [e for e in Path(root_dir).iterdir() if e.is_dir()]
    # Si un seul dossier racine contient tout, on descend d'un niveau
    if len(entries) == 1 and any(sub.is_dir() for sub in entries[0].iterdir()):
        entries = [e for e in entries[0].iterdir() if e.is_dir()]
    return {e.name: str(e) for e in entries}


root = extract_zip(ZIP_PATH, EXTRACT_DIR)
client_folders = list_client_folders(root)
print(f"{len(client_folders)} dossiers clients détectés.")
for cid, path in list(client_folders.items())[:5]:
    print(f"  - client_id={cid}  ->  {path}")


## 3. Repérage flexible des deux PDF cibles dans un dossier client

Tolérant à la casse, aux accents et aux séparateurs (espace/underscore/tiret), pour
absorber les petites variations de nommage d'un pays/back-office à l'autre.


In [ ]:
def normalize_filename(name: str) -> str:
    name = os.path.splitext(name)[0]
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = re.sub(r"[\s_\-]+", "", name.lower())
    return name


def find_target_pdfs(client_folder: str) -> dict:
    """Retourne {"id_card": chemin_ou_None, "residence": chemin_ou_None}."""
    found = {"id_card": None, "residence": None}
    for f in Path(client_folder).glob("*.pdf"):
        norm = normalize_filename(f.name)
        for doc_key, aliases in TARGET_FILES.items():
            if found[doc_key] is not None:
                continue
            if any(normalize_filename(alias) in norm for alias in aliases):
                found[doc_key] = str(f)
    return found


# Exemple de vérification sur l'ensemble des clients
missing_report = []
client_pdfs = {}
for cid, folder in client_folders.items():
    pdfs = find_target_pdfs(folder)
    client_pdfs[cid] = pdfs
    if pdfs["id_card"] is None or pdfs["residence"] is None:
        missing_report.append({"client_id": cid, **{k: (v is not None) for k, v in pdfs.items()}})

print(f"Clients avec un ou deux documents manquants : {len(missing_report)}")
pd.DataFrame(missing_report).head(10)


## 4. Conversion PDF → images (PyMuPDF)

Gère les PDF multi-pages (recto/verso d'une carte d'identité scannée en un seul fichier,
justificatif de résidence sur plusieurs pages, etc.). Rendu à haute résolution pour
préserver la lisibilité des scans de mauvaise qualité.


In [ ]:
def pdf_to_images(pdf_path: str, dpi: int = 300) -> list:
    """Retourne une liste d'images (np.ndarray BGR), une par page."""
    images = []
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(matrix=matrix)
        img_bytes = pix.tobytes("png")
        img_arr = np.frombuffer(img_bytes, dtype=np.uint8)
        img_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        images.append(img_bgr)
    doc.close()
    return images


## 5. Prétraitement image (rotation grossière, skew fin, débruitage/contraste)

In [ ]:
import math

def denoise_and_enhance(img_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def detect_coarse_rotation_osd(img_bgr: np.ndarray) -> int:
    try:
        import pytesseract
        osd = pytesseract.image_to_osd(img_bgr)
        angle = int([l for l in osd.split("\n") if "Rotate" in l][0].split(":")[1].strip())
        return angle
    except Exception as e:
        print(f"[WARN] OSD indisponible ({e}), rotation grossière non détectée.")
        return 0


def rotate_image(img_bgr: np.ndarray, angle: float) -> np.ndarray:
    if angle == 0:
        return img_bgr
    (h, w) = img_bgr.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = np.abs(M[0, 0]); sin = np.abs(M[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    return cv2.warpAffine(img_bgr, M, (new_w, new_h), borderValue=(255, 255, 255))


def detect_and_correct_skew(img_bgr: np.ndarray, max_angle: float = 15.0) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100,
                             minLineLength=gray.shape[1] // 4, maxLineGap=20)
    if lines is None:
        return img_bgr
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
        if -max_angle <= angle <= max_angle:
            angles.append(angle)
    if not angles:
        return img_bgr
    median_angle = float(np.median(angles))
    return img_bgr if abs(median_angle) < 0.5 else rotate_image(img_bgr, median_angle)


def preprocess_image(img_bgr: np.ndarray) -> np.ndarray:
    coarse_angle = detect_coarse_rotation_osd(img_bgr)
    img_bgr = rotate_image(img_bgr, coarse_angle)
    img_bgr = detect_and_correct_skew(img_bgr)
    img_bgr = denoise_and_enhance(img_bgr)
    return img_bgr


## 6. OCR multi-langue adaptatif (PaddleOCR)

Les documents provenant de plusieurs pays, la langue n'est pas connue à l'avance.
Plutôt que de deviner le pays, on **essaie plusieurs packs linguistiques PaddleOCR**
(latin, arabe, cyrillique, CJK) et on conserve le résultat de **plus haute confiance
moyenne** — cela sert avant tout de **score de qualité de scan**, l'extraction de champs
étant confiée aux VLM (plus robustes au multi-langue et à la mise en page).


In [ ]:
from paddleocr import PaddleOCR

PADDLE_LANG_PACKS = ["latin", "arabic", "cyrillic", "ch"]  # "ch" couvre aussi le japonais/coréen en fallback grossier
_paddle_engines = {}

def get_paddle_engine(lang: str) -> PaddleOCR:
    if lang not in _paddle_engines:
        _paddle_engines[lang] = PaddleOCR(use_angle_cls=True, lang=lang, show_log=False)
    return _paddle_engines[lang]


def run_multilingual_ocr(img_bgr: np.ndarray) -> dict:
    """Essaie plusieurs packs linguistiques, garde le meilleur (score de confiance moyen * nb lignes)."""
    best = {"lang": None, "raw_text": "", "avg_confidence": 0.0, "n_lines": 0, "score": -1}
    for lang in PADDLE_LANG_PACKS:
        try:
            engine = get_paddle_engine(lang)
            result = engine.ocr(img_bgr, cls=True)
            lines, scores = [], []
            for page in result:
                if not page:
                    continue
                for box, (text, score) in page:
                    lines.append(text)
                    scores.append(score)
            avg_conf = float(np.mean(scores)) if scores else 0.0
            composite_score = avg_conf * len(lines)
            if composite_score > best["score"]:
                best = {
                    "lang": lang,
                    "raw_text": "\n".join(lines),
                    "avg_confidence": avg_conf,
                    "n_lines": len(lines),
                    "score": composite_score,
                }
        except Exception as e:
            print(f"[WARN] Pack linguistique '{lang}' en échec : {e}")
    return best


## 7. Chargement des VLM (Qwen2.5-VL et InternVL) depuis le ModelHub local

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
from transformers import Qwen2_5_VLForConditionalGeneration

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATHS["qwen_vlm"],
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    local_files_only=True,
)
qwen_processor = AutoProcessor.from_pretrained(MODEL_PATHS["qwen_vlm"], local_files_only=True)

internvl_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATHS["internvl"],
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    local_files_only=True,
    trust_remote_code=True,
)
internvl_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATHS["internvl"], local_files_only=True, trust_remote_code=True
)
print("Modèles chargés.")


## 8. Extraction **schema-free** (aucun champ prédéfini)

Contrairement à la version précédente, on ne fournit **aucune liste de champs** ni de
schéma par type de document. Le modèle doit **lister lui-même** les champs qu'il observe
réellement sur le document, avec leur libellé original et un libellé traduit en français
pour faciliter le rapprochement inter-modèles et inter-documents.


In [ ]:
OPEN_EXTRACTION_PROMPT = """Tu es un expert en analyse de documents d'identité et justificatifs
administratifs, de tous pays et dans toutes les langues.

Observe attentivement ce document (il peut être mal scanné, être une photocopie, légèrement
de travers, ou dans une langue que tu ne maîtrises pas parfaitement).

Ne suppose AUCUN format ni AUCUNE liste de champs prédéfinie : identifie uniquement les
champs réellement visibles sur CE document, tels qu'ils apparaissent.

Réponds UNIQUEMENT avec un objet JSON valide, structure suivante (structure fixe,
mais le CONTENU des champs est entièrement libre) :

{
  "type_document_detecte": "ex: carte d'identite nationale, passeport, titre de sejour, facture, attestation de domicile, ...",
  "pays_emetteur_estime": "",
  "langue_document": "",
  "champs": [
    {"libelle_original": "texte tel qu'il apparaît sur le document", "libelle_normalise_fr": "traduction/normalisation en français, snake_case", "valeur": "valeur lue"},
    ...
  ],
  "remarques": "toute anomalie constatée : illisible, incohérent, incomplet, etc."
}

Liste TOUS les champs que tu peux lire, sans te limiter à un sous-ensemble. Ne donne
aucun texte en dehors du JSON."""


def safe_json_parse(text: str) -> dict:
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return {"_parse_error": True, "_raw_output": text}
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return {"_parse_error": True, "_raw_output": text}


def extract_with_qwen(img_pil: Image.Image) -> dict:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img_pil},
            {"type": "text", "text": OPEN_EXTRACTION_PROMPT},
        ],
    }]
    text_prompt = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text_prompt], images=[img_pil], return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, max_new_tokens=768, do_sample=False)
    output_text = qwen_processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]
    return safe_json_parse(output_text)


def extract_with_internvl(img_pil: Image.Image) -> dict:
    pixel_values = internvl_model.load_image(img_pil) if hasattr(internvl_model, "load_image") else None
    if pixel_values is not None:
        pixel_values = pixel_values.to(internvl_model.device, dtype=internvl_model.dtype)
        response = internvl_model.chat(internvl_tokenizer, pixel_values, OPEN_EXTRACTION_PROMPT, dict(max_new_tokens=768))
    else:
        response = "{}"
    return safe_json_parse(response)


## 9. Réconciliation floue entre les deux VLM (sans schéma commun imposé)

Comme les libellés de champs ne sont plus fixés à l'avance, on les rapproche par
**similarité textuelle** (`difflib`) sur `libelle_normalise_fr`, puis on compare les
valeurs des champs jugés équivalents.


In [ ]:
def normalize_text(v) -> str:
    if v is None:
        return ""
    v = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", v.strip().lower())


def match_fields(champs_a: list, champs_b: list, threshold: float = 0.72) -> list:
    """Apparie les champs de deux extractions par similarité de libellé normalisé."""
    used_b = set()
    matches = []
    for fa in champs_a:
        label_a = normalize_text(fa.get("libelle_normalise_fr"))
        best_j, best_ratio = None, 0.0
        for j, fb in enumerate(champs_b):
            if j in used_b:
                continue
            label_b = normalize_text(fb.get("libelle_normalise_fr"))
            ratio = difflib.SequenceMatcher(None, label_a, label_b).ratio()
            if ratio > best_ratio:
                best_ratio, best_j = ratio, j
        if best_j is not None and best_ratio >= threshold:
            used_b.add(best_j)
            matches.append((fa, champs_b[best_j], best_ratio))
    return matches


def reconcile_extractions(qwen_out: dict, internvl_out: dict) -> dict:
    champs_a = qwen_out.get("champs", []) if isinstance(qwen_out.get("champs"), list) else []
    champs_b = internvl_out.get("champs", []) if isinstance(internvl_out.get("champs"), list) else []

    matches = match_fields(champs_a, champs_b)
    disagreements = []
    agreements = []
    for fa, fb, sim in matches:
        va, vb = normalize_text(fa.get("valeur")), normalize_text(fb.get("valeur"))
        entry = {
            "libelle": fa.get("libelle_normalise_fr"),
            "valeur_qwen": fa.get("valeur"),
            "valeur_internvl": fb.get("valeur"),
            "similarite_libelle": round(sim, 2),
        }
        (agreements if va == vb else disagreements).append(entry)

    n_matched = len(matches)
    n_total = max(len(champs_a), len(champs_b), 1)
    coverage = n_matched / n_total
    agreement_rate = (len(agreements) / n_matched) if n_matched > 0 else 0.0
    confidence = round(coverage * agreement_rate, 3)

    statut = "certifie" if (confidence >= 0.6 and len(disagreements) == 0) else "a_revoir_manuellement"

    return {
        "champs_qwen": champs_a,
        "champs_internvl": champs_b,
        "champs_en_accord": agreements,
        "champs_en_desaccord": disagreements,
        "couverture_appariement": round(coverage, 2),
        "taux_accord": round(agreement_rate, 2),
        "confiance": confidence,
        "statut": statut,
    }


## 10. Cross-vérification d'identité entre `carte identite.pdf` et `residence.pdf`

Sans schéma fixe, on ne peut pas comparer un champ `"nom"` garanti des deux côtés.
On recherche donc, **dans les libellés normalisés**, ceux qui *ressemblent sémantiquement*
à un nom (mots-clés multilingues), et on compare les valeurs correspondantes entre les
deux documents du même client — utile pour détecter une incohérence d'identité.


In [ ]:
NAME_LABEL_HINTS = ["nom", "prenom", "nom complet", "name", "surname", "full name",
                    "nombre", "apellido", "nachname", "vorname", "cognome", "nome"]

def extract_name_like_fields(champs: list) -> list:
    found = []
    for c in champs:
        label = normalize_text(c.get("libelle_normalise_fr")) + " " + normalize_text(c.get("libelle_original"))
        if any(hint in label for hint in NAME_LABEL_HINTS):
            found.append(c.get("valeur"))
    return found


def cross_check_identity(id_card_champs: list, residence_champs: list) -> dict:
    names_id = [normalize_text(n) for n in extract_name_like_fields(id_card_champs) if n]
    names_res = [normalize_text(n) for n in extract_name_like_fields(residence_champs) if n]

    if not names_id or not names_res:
        return {"verifiable": False, "coherent": None, "detail": "champs nominatifs introuvables sur l'un des deux documents"}

    best_ratio = 0.0
    for n1 in names_id:
        for n2 in names_res:
            ratio = difflib.SequenceMatcher(None, n1, n2).ratio()
            best_ratio = max(best_ratio, ratio)

    return {
        "verifiable": True,
        "coherent": best_ratio >= 0.75,
        "similarite_max": round(best_ratio, 2),
        "noms_carte_identite": names_id,
        "noms_residence": names_res,
    }


## 11. Pipeline complet pour un document (PDF → images → prétraitement → double extraction)

In [ ]:
def process_pdf_document(pdf_path: str) -> dict:
    """Traite un PDF (potentiellement multi-pages) : on extrait sur chaque page et on
    fusionne les listes de champs (utile pour recto/verso d'une carte d'identité)."""
    pages = pdf_to_images(pdf_path)
    all_qwen_champs, all_internvl_champs = [], []
    ocr_quality_scores = []
    doc_type_guess, pays_guess, langue_guess = None, None, None

    for page_bgr in pages:
        corrected = preprocess_image(page_bgr)
        img_pil = Image.fromarray(cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB))

        ocr_info = run_multilingual_ocr(corrected)
        ocr_quality_scores.append(ocr_info["avg_confidence"])

        qwen_out = extract_with_qwen(img_pil)
        internvl_out = extract_with_internvl(img_pil)

        all_qwen_champs.extend(qwen_out.get("champs", []) if isinstance(qwen_out.get("champs"), list) else [])
        all_internvl_champs.extend(internvl_out.get("champs", []) if isinstance(internvl_out.get("champs"), list) else [])

        doc_type_guess = doc_type_guess or qwen_out.get("type_document_detecte")
        pays_guess = pays_guess or qwen_out.get("pays_emetteur_estime")
        langue_guess = langue_guess or qwen_out.get("langue_document")

    reconciliation = reconcile_extractions({"champs": all_qwen_champs}, {"champs": all_internvl_champs})

    return {
        "n_pages": len(pages),
        "type_document_detecte": doc_type_guess,
        "pays_emetteur_estime": pays_guess,
        "langue_document": langue_guess,
        "qualite_ocr_moyenne": round(float(np.mean(ocr_quality_scores)), 3) if ocr_quality_scores else 0.0,
        **reconciliation,
    }


## 12. Pipeline complet par client (les deux documents + cross-vérification d'identité)

In [ ]:
def process_client(client_id: str, pdfs: dict) -> dict:
    record = {"client_id": client_id}

    if pdfs.get("id_card") is None:
        record["id_card"] = {"statut": "fichier_absent"}
    else:
        record["id_card"] = process_pdf_document(pdfs["id_card"])

    if pdfs.get("residence") is None:
        record["residence"] = {"statut": "fichier_absent"}
    else:
        record["residence"] = process_pdf_document(pdfs["residence"])

    if pdfs.get("id_card") and pdfs.get("residence"):
        record["cross_verification_identite"] = cross_check_identity(
            record["id_card"].get("champs_qwen", []),
            record["residence"].get("champs_qwen", []),
        )
    else:
        record["cross_verification_identite"] = {"verifiable": False, "detail": "un des deux documents est absent"}

    statuts = [record["id_card"].get("statut"), record["residence"].get("statut")]
    identite_ok = record["cross_verification_identite"].get("coherent", True)
    if "fichier_absent" in statuts or "a_revoir_manuellement" in statuts or identite_ok is False:
        record["statut_global"] = "a_revoir_manuellement"
    else:
        record["statut_global"] = "certifie"

    return record


## 13. Traitement par lot sur l'ensemble des clients du ZIP

In [ ]:
def batch_process_zip(zip_path: str) -> pd.DataFrame:
    root = extract_zip(zip_path, EXTRACT_DIR)
    folders = list_client_folders(root)
    print(f"{len(folders)} clients détectés dans l'archive.")

    all_records = []
    for cid, folder in folders.items():
        pdfs = find_target_pdfs(folder)
        try:
            rec = process_client(cid, pdfs)
        except Exception as e:
            rec = {"client_id": cid, "statut_global": "erreur_traitement", "erreur": str(e)}
        all_records.append(rec)
        print(f"client_id={cid:15s} -> statut_global = {rec.get('statut_global')}")

    df = pd.DataFrame(all_records)
    return df, all_records


# Exemple d'exécution :
# df_resultats, records_bruts = batch_process_zip(ZIP_PATH)
# df_resultats.to_csv(f"{OUTPUT_DIR}/resultats_recertification_clients.csv", index=False)
# with open(f"{OUTPUT_DIR}/resultats_recertification_clients.json", "w", encoding="utf-8") as f:
#     json.dump(records_bruts, f, ensure_ascii=False, indent=2)
# df_resultats


## Notes de production

- **Robustesse au nommage** : si les fichiers ne sont pas strictement nommés `carte identite.pdf` / `residence.pdf`, élargir la liste `TARGET_FILES` avec d'autres alias observés en production (multi-langues : `national_id`, `personalausweis`, `documento_identidad`, `justificatif_domicile`, `proof_of_address`, etc.).
- **Absence de schéma** : la structure `{"champs": [{"libelle_original", "libelle_normalise_fr", "valeur"}]}` reste volontairement minimale — elle n'impose aucune liste de champs attendus, seulement une forme d'encapsulation exploitable en aval (base de données, reporting).
- **Cross-vérification d'identité** : le rapprochement par mots-clés multilingues (`NAME_LABEL_HINTS`) est un heuristique de départ ; à enrichir avec les libellés réellement rencontrés lors des premiers lots traités.
- **Traçabilité KYC** : conserver systématiquement le JSON brut de chaque page (Qwen + InternVL) et le score `qualite_ocr_moyenne`, même en cas de certification automatique, pour audit.
- **Performance** : pour de gros volumes de clients, paralléliser `process_client` (multiprocessing / batch GPU) et envisager un pré-filtre léger (SmolVLM) pour écarter rapidement les scans totalement illisibles avant d'engager Qwen2.5-VL/InternVL.
